In [13]:
# =========================================
# Cell 1 — Install dependencies
# =========================================
!pip -q install datasets>=2.19.0 anthropic>=0.35.0 pandas>=2.2.0 numpy>=1.26.0 scikit-learn>=1.3.0 tqdm>=4.66.0 matplotlib>=3.8.0


In [14]:
# =========================================
# Cell 2 — Keys (type them in)
# =========================================
import os
import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter ANTHROPIC_API_KEY (input hidden): ")

hf = getpass.getpass("Enter HF_TOKEN (optional, input hidden; press Enter to skip): ")
if hf.strip():
    os.environ["HF_TOKEN"] = hf.strip()

assert os.environ.get("ANTHROPIC_API_KEY"), "Missing ANTHROPIC_API_KEY"
print("✅ Keys set. HF token provided?", bool(os.environ.get("HF_TOKEN")))


Enter ANTHROPIC_API_KEY (input hidden): ··········
Enter HF_TOKEN (optional, input hidden; press Enter to skip): ··········
✅ Keys set. HF token provided? True


In [15]:
# =========================================
# Cell 3 — Load FPB dataset from Hugging Face
# =========================================
from datasets import load_dataset

DATASET_NAME = "TheFinAI/en-fpb"
SPLIT = "test"   # change if needed (e.g., "train")

ds = load_dataset(DATASET_NAME, split=SPLIT, token=os.environ.get("HF_TOKEN", None))
print(ds)
print("Columns:", ds.column_names)
print("Example row:\n", ds[0])


Dataset({
    features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
    num_rows: 970
})
Columns: ['id', 'query', 'answer', 'text', 'choices', 'gold']
Example row:
 {'id': 'fpb3876', 'query': 'Analyze the sentiment of this statement extracted from a financial news article. Provide your answer as either negative, positive, or neutral.\nText: The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .\nAnswer:', 'answer': 'positive', 'text': 'The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .', 'choices': ['positive', 'neutral', 'negative'], 'gold': 0}


In [16]:
# =========================================
# Cell 4 — Prompt + Claude API call helpers
# =========================================
import re
import time
from anthropic import Anthropic

MODEL_ID = "claude-sonnet-4-20250514"  # Claude Sonnet 4 API ID :contentReference[oaicite:1]{index=1}
client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

def build_prompt(text: str, choices):
    # Keep it strict so parsing is easy
    choices_str = ", ".join([str(c) for c in choices])
    return (
        "Analyze the sentiment of this financial text.\n"
        "Respond ONLY with one label from the Options.\n\n"
        f"Text: {text}\n"
        f"Options: {choices_str}\n"
        "Label:"
    )

def extract_label(model_text: str, choices):
    if model_text is None:
        return None
    s = model_text.strip().lower()

    # exact match
    for c in choices:
        if s == str(c).strip().lower():
            return str(c)

    # search for whole-word labels
    for c in choices:
        pat = r"\b" + re.escape(str(c).strip().lower()) + r"\b"
        if re.search(pat, s):
            return str(c)

    return None

def claude_predict(prompt: str, model: str = MODEL_ID, max_tokens: int = 8, temperature: float = 0.0,
                  max_retries: int = 5, sleep_base: float = 1.0):
    # Messages API usage: :contentReference[oaicite:2]{index=2}
    last_err = None
    for attempt in range(max_retries):
        try:
            t0 = time.time()
            msg = client.messages.create(
                model=model,
                max_tokens=max_tokens,
                temperature=temperature,
                system="You are a financial analyst.",
                messages=[{"role": "user", "content": prompt}],
            )
            latency = time.time() - t0

            # Claude returns content as a list of blocks; usually first is text
            text_out = ""
            if msg and getattr(msg, "content", None):
                # concatenate any text blocks
                parts = []
                for block in msg.content:
                    if getattr(block, "type", None) == "text":
                        parts.append(block.text)
                text_out = "".join(parts).strip()

            usage = getattr(msg, "usage", None)
            usage_dict = None
            if usage:
                usage_dict = {
                    "input_tokens": getattr(usage, "input_tokens", None),
                    "output_tokens": getattr(usage, "output_tokens", None),
                }

            return text_out, {"latency_s": latency, "usage": usage_dict, "stop_reason": getattr(msg, "stop_reason", None)}
        except Exception as e:
            last_err = str(e)
            time.sleep(sleep_base * (2 ** attempt))
    return None, {"error": last_err}


In [17]:
# =========================================
# Cell 5 — Run evaluation (sampled or full) + save raw outputs
# =========================================
import json
import random
from tqdm import tqdm
import os

# ---- controls ----
SEED = 42
MAX_SAMPLES = None   # set to None to run the full split
SHUFFLE = True      # random sample if True, else take first MAX_SAMPLES
SLEEP_BETWEEN_CALLS = 0.1  # gentle throttling

random.seed(SEED)

# ---- output dirs (matches your pipeline layout idea) ----
os.makedirs("text_responses", exist_ok=True)
os.makedirs("evaluation", exist_ok=True)

run_tag = f"claude_sonnet4_{DATASET_NAME.replace('/','_')}_{SPLIT}"
raw_path = f"text_responses/{run_tag}.jsonl"

# choose indices
idxs = list(range(len(ds)))
if SHUFFLE:
    random.shuffle(idxs)
if MAX_SAMPLES is not None:
    idxs = idxs[:MAX_SAMPLES]

records = []
pred_indices = []
gold_indices = []

with open(raw_path, "w", encoding="utf-8") as f:
    for i in tqdm(idxs, desc="Evaluating"):
        ex = ds[i]
        text = ex["text"]
        choices = ex["choices"]            # e.g. ["positive","neutral","negative"]
        gold = int(ex["gold"])             # usually 0/1/2

        prompt = build_prompt(text, choices)
        model_out, meta = claude_predict(prompt)

        pred_label = extract_label(model_out, choices)
        pred_idx = choices.index(pred_label) if pred_label in choices else -1

        rec = {
            "row_index": i,
            "id": ex.get("id", None),
            "text": text,
            "choices": choices,
            "gold": gold,
            "model_response_raw": model_out,
            "predicted_label": pred_label,
            "predicted_index": pred_idx,
            "correct": (pred_idx == gold),
            "meta": meta,
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

        records.append(rec)
        gold_indices.append(gold)
        pred_indices.append(pred_idx)

        if SLEEP_BETWEEN_CALLS:
            time.sleep(SLEEP_BETWEEN_CALLS)

print("✅ Saved raw responses to:", raw_path)
print("Samples evaluated:", len(records))


Evaluating: 100%|██████████| 970/970 [44:02<00:00,  2.72s/it]

✅ Saved raw responses to: text_responses/claude_sonnet4_TheFinAI_en-fpb_test.jsonl
Samples evaluated: 970


In [18]:
# =========================================
# Cell 6 — Compute metrics + save evaluation artifacts
# =========================================
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

df = pd.DataFrame(records)

# Filter out invalid predictions (-1) for some metrics if you want:
valid_mask = df["predicted_index"] >= 0

acc_all = accuracy_score(df["gold"], df["predicted_index"])
acc_valid = accuracy_score(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"]) if valid_mask.any() else None

labels_sorted = sorted(list(set(df["gold"].tolist())))  # usually [0,1,2]

metrics = {
    "run_tag": run_tag,
    "model_id": MODEL_ID,
    "dataset": DATASET_NAME,
    "split": SPLIT,
    "num_samples": int(len(df)),
    "num_valid_predictions": int(valid_mask.sum()),
    "accuracy_all": float(acc_all),
    "accuracy_valid_only": (float(acc_valid) if acc_valid is not None else None),
    "f1_macro_valid_only": float(f1_score(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"],
                                         average="macro", labels=labels_sorted)) if valid_mask.any() else None,
    "f1_micro_valid_only": float(f1_score(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"],
                                         average="micro", labels=labels_sorted)) if valid_mask.any() else None,
    "f1_weighted_valid_only": float(f1_score(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"],
                                            average="weighted", labels=labels_sorted)) if valid_mask.any() else None,
}

cm = confusion_matrix(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"], labels=labels_sorted) if valid_mask.any() else None

# Save metrics JSON
metrics_path = f"evaluation/{run_tag}_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump({"metrics": metrics, "confusion_matrix_labels": labels_sorted, "confusion_matrix": (cm.tolist() if cm is not None else None)},
              f, ensure_ascii=False, indent=2)

# Save per-example CSV
preds_path = f"evaluation/{run_tag}_predictions.csv"
df.to_csv(preds_path, index=False)

print("✅ Metrics saved to:", metrics_path)
print("✅ Predictions saved to:", preds_path)

# Print a readable report (valid preds only)
if valid_mask.any():
    print("\nClassification report (valid predictions only):")
    print(classification_report(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"], labels=labels_sorted))
else:
    print("No valid predictions to report.")


✅ Metrics saved to: evaluation/claude_sonnet4_TheFinAI_en-fpb_test_metrics.json
✅ Predictions saved to: evaluation/claude_sonnet4_TheFinAI_en-fpb_test_predictions.csv

Classification report (valid predictions only):
              precision    recall  f1-score   support

           0       0.65      0.88      0.75       277
           1       0.93      0.71      0.81       577
           2       0.77      0.98      0.86       116

    accuracy                           0.79       970
   macro avg       0.78      0.86      0.81       970
weighted avg       0.83      0.79      0.80       970

